In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['GROQ_API_KEY']:
    print("Groq API Key Set!")
else:
    raise ValueError("Groq API Key is not set!!")

Groq API Key Set!


In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(model = "llama-3.1-8b-instant")

llm.invoke("Hello!").content    

'How can I assist you today?'

## **TOOLS**

### Duck Duck Go Search Tool

In [36]:
from langchain_community.tools import DuckDuckGoSearchRun

duck_search = DuckDuckGoSearchRun()

duck_search.invoke("Latest Hot News on Iran-US War?")

"46 minutes ago - A US official said dozens of US strikes were carried out by planes based around the Middle East and from one or more aircraft carriers. According to Iran International, quoting the Iranian Students' News Agency, thousands of Islamic Revolutionary Guard Corps (IRGC) personnel, including several senior officials, were killed or wounded as several military bases were attacked. 12 hours ago - Exploring the unintended consequences of the US-Iran conflict for global defense and security, including hybrid threats, technology, alliances, and strategy. ... A fragile ceasefire masks a far more dangerous reality: Iran’s nuclear ambitions unresolved, Lebanon destabilized, terrorism risks rising, and a shadow war between Israel and Iran poised to reignite at any moment. 13 hours ago - U.S. President Donald Trump announced on Thursday that Lebanon and Israel had agreed on a 10-day ceasefire, as optimism grew that the Iran war may be nearing an end. 46 minutes ago - US-Iran-Israel W

### Arxiv - Find Research Papers

In [10]:
from langchain_community.retrievers import ArxivRetriever

retriever = ArxivRetriever(
    load_max_docs=2,
    get_full_documents=True,
)

retriever.invoke("Transformer in NLP")

[Document(metadata={'Published': '2021-05-14', 'Title': 'A dissemination workshop for introducing young Italian students to NLP', 'Authors': 'Lucio Messina, Lucia Busso, Claudia Roberta Combei, Ludovica Pannitto, Alessio Miaschi, Gabriele Sarti, Malvina Nissim', 'Summary': 'We describe and make available the game-based material developed for a laboratory run at several Italian science festivals to popularize NLP among young students.'}, page_content='A dissemination workshop for introducing young Italian students to NLP\nLucio Messina\nIndependent Researcher\nlucio.messina@autistici.org\nLucia Busso\nAston University\nl.busso@aston.ac.uk\nClaudia Roberta Combei\nUniversity of Bologna\nclaudiaroberta.combei@unibo.it\nLudovica Pannitto\nUniversity of Trento\nludovica.pannitto@unitn.it\nAlessio Miaschi\nUniversity of Pisa\nalessio.miaschi@phd.unipi.it\nGabriele Sarti\nUniversity of Trieste\ngsarti@sissa.it\nMalvina Nissim\nUniversity of Groningen\nm.nissim@rug.nl\nAbstract\nWe describe an

In [11]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())

arxiv_query.invoke("Transformers in NLP")

'Published: 2021-09-23\nTitle: Transformers: "The End of History" for NLP?\nAuthors: Anton Chernyavskiy, Dmitry Ilvovsky, Preslav Nakov\nSummary: Recent advances in neural architectures, such as the Transformer, coupled with the emergence of large-scale pre-trained models such as BERT, have revolutionized the field of Natural Language Processing (NLP), pushing the state of the art for a number of NLP tasks. A rich family of variations of these models has been proposed, such as RoBERTa, ALBERT, and XLNet, but fundamentally, they all remain limited in their ability to model certain kinds of information, and they cannot cope with certain information sources, which was easy for pre-existing models. Thus, here we aim to shed light on some important theoretical limitations of pre-trained BERT-style models that are inherent in the general Transformer architecture. First, we demonstrate in practice on two general types of tasks -- segmentation and segment labeling -- and on four datasets that 

### Wikipedia Search Tool

In [13]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

wikipedia.invoke("What are AI Agent Frameworks?")

'Page: AI agent\nSummary: In the context of generative artificial intelligence, AI agents (also referred to as compound AI systems or agentic AI) are a class of intelligent agents distinguished by their ability to operate autonomously in complex environments. Agentic AI tools prioritize decision-making over content creation and do not require continuous oversight.\n\n\n\nPage: Intelligent agent\nSummary: In artificial intelligence, an intelligent agent is an entity that perceives its environment, takes actions autonomously to achieve goals, and may improve its performance through machine learning or by acquiring knowledge. AI textbooks define artificial intelligence as the "study and design of intelligent agents," emphasizing that goal-directed behavior is central to intelligence.\nA specialized subset of intelligent agents, agentic AI (also known as an AI agent or simply agent), expands this concept by proactively pursuing goals, making decisions, and taking actions over extended peri

### Custom Tool

In [25]:
from langchain.tools import tool

@tool
def personal_tool(name:str):

    """Use this tool to get personal information about Alice, Bob & Charlie!"""

    info = {
        "Alice" : "Alice is Software Developer with 5 yrs experience in AI",
        "Bob" : "Bob is a data scientist who loves working with large datasets",
        "Charlie" : "Charlie is Product Manager with background in tech startups"
    }

    return info.get(name,"No information about this person!")

In [18]:
personal_tool.invoke("Alice")

'Alice is Software Developer with 5 yrs experience in AI'

In [19]:
personal_tool.invoke("Harsh")

'No information about this person!'

## **TOOL BINDING**

In [23]:
tools = [duck_search, arxiv_query, wikipedia, personal_tool]

llm_with_tools = llm.bind_tools(tools)

In [32]:
response = llm_with_tools.invoke("Latest News on AI")

response.tool_calls

[{'name': 'duckduckgo_search',
  'args': {'query': 'AI latest news'},
  'id': '6n05ssycj',
  'type': 'tool_call'}]

In [34]:
response2 = llm_with_tools.invoke("I want to know personal information about Alice?")

response2.tool_calls

[{'name': 'personal_tool',
  'args': {'name': 'Alice'},
  'id': '5f50q1jf3',
  'type': 'tool_call'}]